# 2022 Inventory Rerun Pipeline with Checkpointing & Model Traceability

**Purpose**: Reprocess 2022 EuropePMC dataset with latest production models using GPU acceleration and checkpoint recovery  
**Created**: 2025-10-23  
**Updated**: 2025-10-24 (Restructured with utility functions and fixed critical bugs)  
**Environment**: Google Colab with GPU support  
**Status**: Production Pipeline with Hybrid Checkpointing & Mandatory Model Traceability  

---

## Overview

This notebook processes the 2022 EuropePMC dataset (21,677 papers) through a streamlined 5-step pipeline with:

- **Mandatory Model Traceability** - TRAINING_SESSION_ID required for full audit trail
- **Streamlined Processing** - Optimized for fixed 2022 dataset
- **GPU Acceleration** - 5-10x faster inference
- **Hybrid Checkpointing** - Resume from failed runs
- **Google Drive Backup** - All major computational steps backed up
- **Clean Architecture** - Uses `src/rerun_utils.py` for all utility functions

## Pipeline Steps

1. **Input Validation** → Verify 2022 dataset and models
2. **Classification** → Identify bio-resource papers
3. **Named Entity Recognition** → Extract database names
4. **URL Extraction** → Find resource URLs
5. **Name Processing** → Generate final inventory

## Key Features

- ✅ **Fixed Critical Bugs**: Script paths use absolute paths with INVENTORY_DIRECTORY
- ✅ **Archive Path Fix**: Uses correct `training_archives/{ID}_full_training/` path
- ✅ **Utility Functions**: All functions extracted to `src/rerun_utils.py`
- ✅ **Mandatory Traceability**: No fallback to production models
- ✅ **Clean Cell Structure**: 12 cells following training notebook pattern

In [ ]:
# =============================================================================
# CELL 1: MOUNT GOOGLE DRIVE
# =============================================================================
# IMPORTANT: This MUST be the first cell for checkpoint access

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully")
print("💾 Checkpoint and archive paths are now accessible")
print("🔗 Ready for hybrid checkpointing system")

In [ ]:
# =============================================================================
# CELL 2: RERUN PIPELINE CONFIGURATION
# =============================================================================

import os
import random
import string
from datetime import datetime

# =============================================================================
# USER-EDITABLE CONFIGURATION
# =============================================================================

# REQUIRED: Model Traceability - Link to specific training session
TRAINING_SESSION_ID = ""  # e.g., "2025-10-23-abc123" from training notebook
# This MUST match the UNIQUE_ID from your training notebook run

# Input Data Configuration
INPUT_DATA = "data/epmc_query_results_2022.csv"  # Fixed: 2022 dataset
RUN_MODE = "full"  # Options: "full" (21,677 papers) or "test" (subset)
TEST_SUBSET_SIZE = 1000  # Papers to process in test mode

# Processing Configuration
MAX_URLS = 3  # Maximum URLs to extract per paper

# =============================================================================
# AUTO-GENERATED CONFIGURATION (DO NOT EDIT BELOW THIS LINE)
# =============================================================================

# Session Management
RERUN_SESSION_ID = f"{datetime.now().strftime('%Y-%m-%d')}-{''.join(random.choices(string.ascii_lowercase + string.digits, k=6))}"
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
RUN_DATE = datetime.now().strftime('%Y-%m-%d')

# Path Configuration
INVENTORY_DIRECTORY = "/content/drive/MyDrive/inventory_2022"
DATA_DIRECTORY = f"{INVENTORY_DIRECTORY}/data"

# Checkpoint Configuration
CHECKPOINT_BASE = f"{INVENTORY_DIRECTORY}/rerun_checkpoints/{RERUN_SESSION_ID}"
USE_CHECKPOINTS = True

# Model Archive Paths (FIXED to match training notebook)
TRAINING_ARCHIVE_BASE = f"{INVENTORY_DIRECTORY}/training_archives/{TRAINING_SESSION_ID}_full_training"
ARCHIVE_CLASSIF_MODEL = f"{TRAINING_ARCHIVE_BASE}/classification_model.pt"
ARCHIVE_NER_MODEL = f"{TRAINING_ARCHIVE_BASE}/ner_model.pt"

# Working Model Locations (where scripts expect them)
TARGET_CLASSIF_MODEL = "out/classif_train_out/article_classifier.pt"
TARGET_NER_MODEL = "out/ner_train_out/named_entity_recognition.pt"

# Output Configuration
OUTPUT_BASE_DIR = "inventory_classification_results"
OUTPUT_RUN_DIR = f"{OUTPUT_BASE_DIR}/{RUN_DATE}_2022_rerun"
CLASSIF_DIR = f"{OUTPUT_RUN_DIR}/classification"
NER_DIR = f"{OUTPUT_RUN_DIR}/ner"
URL_DIR = f"{OUTPUT_RUN_DIR}/url_extraction"
NAMES_DIR = f"{OUTPUT_RUN_DIR}/processed_names"
FINAL_DIR = f"{OUTPUT_RUN_DIR}/final_results"
LOG_DIR = f"{OUTPUT_RUN_DIR}/logs"

# Results Archive
RESULTS_ARCHIVE_BASE = f"{INVENTORY_DIRECTORY}/rerun_results/{RERUN_SESSION_ID}_2022_rerun"

# Results file paths
CLASSIF_RESULTS = f"{CLASSIF_DIR}/predictions.csv"
CLASSIF_POSITIVES = f"{CLASSIF_DIR}/predicted_positives.csv"
NER_RESULTS = f"{NER_DIR}/predictions.csv"
URL_RESULTS = f"{URL_DIR}/predictions.csv"
NAMES_RESULTS = f"{NAMES_DIR}/predictions.csv"
FINAL_RESULTS = f"{FINAL_DIR}/biodata_inventory_2022_rerun.csv"

# Environment
os.environ['PYTHONPATH'] = 'src'

# Configuration dictionary for utilities
config = {
    'rerun_session_id': RERUN_SESSION_ID,
    'training_session_id': TRAINING_SESSION_ID,
    'timestamp': TIMESTAMP,
    'run_mode': RUN_MODE,
    'test_subset_size': TEST_SUBSET_SIZE,
    'input_data': INPUT_DATA,
    'max_urls': MAX_URLS,
    'inventory_directory': INVENTORY_DIRECTORY,
    'data_directory': DATA_DIRECTORY,
    'checkpoint_base': CHECKPOINT_BASE,
    'training_archive_base': TRAINING_ARCHIVE_BASE,
    'results_archive_base': RESULTS_ARCHIVE_BASE,
    'created': datetime.now().isoformat()
}

print("=" * 60)
print("CONFIGURATION LOADED")
print("=" * 60)

In [ ]:
# =============================================================================
# CELL 3: VALIDATION & DISPLAY
# =============================================================================

import sys
from pathlib import Path

# Setup Python path
if f'{INVENTORY_DIRECTORY}/' not in sys.path:
    sys.path.append(f'{INVENTORY_DIRECTORY}/')

# Import utilities
from src.rerun_utils import validate_rerun_config, display_rerun_config

# Validate configuration
validate_rerun_config(config)

# Display configuration
display_rerun_config(config)

print("\n" + "=" * 60)
print("CONFIGURATION VALIDATED")
print("=" * 60)

In [ ]:
# =============================================================================
# CELL 4: ENVIRONMENT SETUP
# =============================================================================

print("🔧 Installing packages...")
!pip install transformers datasets evaluate seqeval nltk

print("\n✅ Dependencies installed")

# Download NLTK data
import nltk
import ssl

print("Downloading NLTK data...")
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt_tab')
print("✅ NLTK data downloaded")

# Import all utilities
from src.rerun_utils import *

# Import libraries
import pandas as pd
import numpy as np
import torch
import transformers
import shutil
import time
import json

# Verify environment
print(f"\n🐍 Python: {sys.version.split()[0]}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🤗 Transformers: {transformers.__version__}")
print(f"🎯 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")

print("\n✅ Environment setup complete")

In [ ]:
# =============================================================================
# CELL 5: GPU CHECK
# =============================================================================

print("🔍 GPU Environment Check:")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print("✅ GPU acceleration available")
else:
    print("⚠️ No GPU detected - inference will be slower")
    print("Consider enabling GPU runtime: Runtime > Change runtime type > GPU")

In [ ]:
# =============================================================================
# CELL 6: CHECKPOINT SYSTEM SETUP
# =============================================================================

print("🔧 Setting up checkpoint system...")

# Create checkpoint directory
Path(CHECKPOINT_BASE).mkdir(parents=True, exist_ok=True)

# Save configuration
with open(f"{CHECKPOINT_BASE}/config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ Config saved: {CHECKPOINT_BASE}/config.json")

# Initialize progress tracker
progress = {
    "input_validation": "⏳ pending",
    "classification": "⏳ pending",
    "ner_processing": "⏳ pending",
    "url_extraction": "⏳ pending",
    "final_processing": "⏳ pending"
}

show_rerun_progress(progress, RERUN_SESSION_ID)
print("\n✅ Checkpoint system ready!")

In [ ]:
# =============================================================================
# CELL 7: MODEL LOADING WITH TRACEABILITY
# =============================================================================

print("🤖 Model Loading & Traceability System")
print("=" * 50)

# Create target directories
Path("out/classif_train_out").mkdir(parents=True, exist_ok=True)
Path("out/ner_train_out").mkdir(parents=True, exist_ok=True)

# Load models with traceability
model_source, training_session_used = load_models_with_traceability(
    TRAINING_SESSION_ID,
    INVENTORY_DIRECTORY,
    TARGET_CLASSIF_MODEL,
    TARGET_NER_MODEL
)

# Update config with traceability
config['model_traceability'] = {
    'model_source': model_source,
    'training_session_used': training_session_used,
    'models_loaded_at': datetime.now().isoformat()
}

# Save updated config
with open(f"{CHECKPOINT_BASE}/config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n🎯 Model Traceability Established:")
print(f"   Training Session → {training_session_used}")
print(f"   Rerun Session → {RERUN_SESSION_ID}")
print(f"   Model Source → {model_source}")
print("\n✅ Models ready for inference!")

In [ ]:
# =============================================================================
# CELL 8: INPUT VALIDATION
# =============================================================================

print("📚 Input Data Validation")
print("=" * 50)

# Create output directories
output_dirs = [OUTPUT_RUN_DIR, CLASSIF_DIR, NER_DIR, URL_DIR, NAMES_DIR, FINAL_DIR, LOG_DIR]
for dir_path in output_dirs:
    Path(dir_path).mkdir(parents=True, exist_ok=True)

# Validate input data (2022 dataset uses 'id' and 'abstract' column names)
valid, total_papers, columns = validate_input_data(
    f"{DATA_DIRECTORY}/{INPUT_DATA.split('/')[-1]}",
    ['id', 'title', 'abstract']
)

if not valid:
    raise ValueError(f"Input data validation failed for {INPUT_DATA}")

print(f"✅ Input data validated: {total_papers:,} papers")
print(f"📋 Columns: {columns}")

# Handle test mode
if RUN_MODE == "test":
    print(f"\n🧪 Test Mode: Using subset of {TEST_SUBSET_SIZE} papers")
    input_df = pd.read_csv(f"{DATA_DIRECTORY}/{INPUT_DATA.split('/')[-1]}").head(TEST_SUBSET_SIZE)
    test_input_path = f"{OUTPUT_RUN_DIR}/test_input.csv"
    input_df.to_csv(test_input_path, index=False)
    effective_input = test_input_path
    papers_to_process = len(input_df)
    print(f"📁 Test subset saved: {test_input_path}")
else:
    effective_input = f"{DATA_DIRECTORY}/{INPUT_DATA.split('/')[-1]}"
    papers_to_process = total_papers
    print(f"\n🔍 Full Mode: Processing all {papers_to_process:,} papers")

progress["input_validation"] = "✅ completed"
show_rerun_progress(progress, RERUN_SESSION_ID)
print("\n✅ Input validation complete")

In [ ]:
# =============================================================================
# CELL 9: CLASSIFICATION PIPELINE
# =============================================================================

progress['classification'] = '🔄 running'
show_rerun_progress(progress, RERUN_SESSION_ID)

print("📋 Step 1/5: Classification Pipeline")
print("=" * 40)

# Check existing results
local_exists, local_count = check_local_results(CLASSIF_RESULTS)

if local_exists:
    print(f"✅ Local classification results found ({local_count:,} papers)")
    progress['classification'] = '✅ loaded from local'

elif USE_CHECKPOINTS and check_drive_checkpoint(CHECKPOINT_BASE, 'classification'):
    print("📥 Loading from checkpoint...")
    load_step_from_checkpoint(CHECKPOINT_BASE, 'classification', CLASSIF_DIR)
    df = pd.read_csv(CLASSIF_RESULTS)
    print(f"✅ Loaded {len(df):,} predictions from checkpoint")
    progress['classification'] = '✅ loaded from checkpoint'

else:
    print("🚀 Running classification...")
    print(f"📅 Input: {effective_input}")
    print(f"🤖 Model: {TARGET_CLASSIF_MODEL}")

    start_time = time.time()

    # Run classification
    success, duration = run_prediction_script(
        'class_predict',
        INVENTORY_DIRECTORY,
        {
            '-i': effective_input,
            '-o': CLASSIF_DIR,
            '-c': TARGET_CLASSIF_MODEL
        }
    )

    if success:
        # Filter positives
        df_all = pd.read_csv(CLASSIF_RESULTS)
        positives = df_all[df_all['predicted_label'] == 'bio-resource']
        positives.to_csv(CLASSIF_POSITIVES, index=False)

        duration_mins = int(duration // 60)
        duration_secs = int(duration % 60)
        print(f"✅ Classification completed in {duration_mins}m {duration_secs}s")
        print(f"📊 Total: {len(df_all):,}, Bio-resource: {len(positives):,} ({len(positives)/len(df_all)*100:.1f}%)")

        # Save to checkpoint
        if USE_CHECKPOINTS:
            save_step_to_checkpoint(CHECKPOINT_BASE, 'classification', CLASSIF_DIR)
            print("💾 Saved to checkpoint")

        progress['classification'] = '✅ completed'
    else:
        raise RuntimeError("Classification processing failed")

show_rerun_progress(progress, RERUN_SESSION_ID)
print("🎯 Classification step complete!")

In [ ]:
# =============================================================================
# CELL 10: NER PIPELINE
# =============================================================================

progress['ner_processing'] = '🔄 running'
show_rerun_progress(progress, RERUN_SESSION_ID)

print("🏷️ Step 2/5: NER Pipeline")
print("=" * 40)

# Check existing results
local_exists, local_count = check_local_results(NER_RESULTS)

if local_exists:
    print(f"✅ Local NER results found ({local_count:,} papers)")
    progress['ner_processing'] = '✅ loaded from local'

elif USE_CHECKPOINTS and check_drive_checkpoint(CHECKPOINT_BASE, 'ner_processing'):
    print("📥 Loading from checkpoint...")
    load_step_from_checkpoint(CHECKPOINT_BASE, 'ner_processing', NER_DIR)
    df = pd.read_csv(NER_RESULTS)
    print(f"✅ Loaded {len(df):,} NER results from checkpoint")
    progress['ner_processing'] = '✅ loaded from checkpoint'

else:
    print("🚀 Running NER...")
    print(f"📅 Input: {CLASSIF_POSITIVES}")
    print(f"🤖 Model: {TARGET_NER_MODEL}")

    # Verify input exists
    if not Path(CLASSIF_POSITIVES).exists():
        raise FileNotFoundError(f"Bio-resource papers file not found: {CLASSIF_POSITIVES}")

    input_df = pd.read_csv(CLASSIF_POSITIVES)
    print(f"📊 Bio-resource papers to process: {len(input_df):,}")

    # Run NER
    success, duration = run_prediction_script(
        'ner_predict',
        INVENTORY_DIRECTORY,
        {
            '-i': CLASSIF_POSITIVES,
            '-o': NER_DIR,
            '-c': TARGET_NER_MODEL
        }
    )

    if success:
        df = pd.read_csv(NER_RESULTS)
        duration_mins = int(duration // 60)
        duration_secs = int(duration % 60)
        print(f"✅ NER completed in {duration_mins}m {duration_secs}s")
        print(f"📊 Papers with NER results: {len(df):,}")

        # Save to checkpoint
        if USE_CHECKPOINTS:
            save_step_to_checkpoint(CHECKPOINT_BASE, 'ner_processing', NER_DIR)
            print("💾 Saved to checkpoint")

        progress['ner_processing'] = '✅ completed'
    else:
        raise RuntimeError("NER processing failed")

show_rerun_progress(progress, RERUN_SESSION_ID)
print("🎯 NER step complete!")

In [ ]:
# =============================================================================
# CELL 11: POST-PROCESSING PIPELINE (URL EXTRACTION + NAME PROCESSING)
# =============================================================================

print("🔍 Step 3-4/5: Post-Processing Pipeline")
print("=" * 40)

# URL Extraction
print("\n📋 Step 3/5: URL Extraction")
url_exists, url_count = check_local_results(URL_RESULTS)

if url_exists:
    print(f"✅ URL extraction results found ({url_count:,} entries)")
else:
    print("🚀 Starting URL extraction...")
    success, duration = run_prediction_script(
        'url_extractor',
        INVENTORY_DIRECTORY,
        {
            NER_RESULTS: None,  # positional arg
            '-o': URL_DIR,
            '-x': str(MAX_URLS)
        }
    )

    if success:
        df = pd.read_csv(URL_RESULTS)
        print(f"✅ URL extraction completed in {duration:.1f}s")
        print(f"📊 Papers with URLs: {len(df):,}")
    else:
        raise RuntimeError("URL extraction failed")

progress["url_extraction"] = "✅ completed"

# Name Processing
print("\n📋 Step 4/5: Name Processing")
names_exists, names_count = check_local_results(NAMES_RESULTS)

if names_exists:
    print(f"✅ Name processing results found ({names_count:,} entries)")
else:
    print("🚀 Starting name processing...")
    success, duration = run_prediction_script(
        'process_names',
        INVENTORY_DIRECTORY,
        {
            URL_RESULTS: None,  # positional arg
            '-o': NAMES_DIR
        }
    )

    if success:
        df = pd.read_csv(NAMES_RESULTS)
        print(f"✅ Name processing completed in {duration:.1f}s")
        print(f"📊 Processed entries: {len(df):,}")
    else:
        raise RuntimeError("Name processing failed")

show_rerun_progress(progress, RERUN_SESSION_ID)
print("🎯 Post-processing steps complete!")

In [ ]:
# =============================================================================
# CELL 12: FINAL RESULTS & COMPREHENSIVE ARCHIVE
# =============================================================================

print("🎉 Step 5/5: Final Results Creation & Archive")
print("=" * 40)

# Create final inventory
print("\n📋 Creating final inventory...")
if Path(NAMES_RESULTS).exists():
    shutil.copy2(NAMES_RESULTS, FINAL_RESULTS)
    final_df = pd.read_csv(FINAL_RESULTS)
    final_count = len(final_df)
    print(f"✅ Final inventory created: {final_count:,} biodata resources")
    print(f"📁 Location: {FINAL_RESULTS}")
else:
    raise FileNotFoundError(f"Name processing results not found: {NAMES_RESULTS}")

progress["final_processing"] = "✅ completed"

# Create comprehensive archive
print(f"\n💾 Creating comprehensive archive...")
print(f"📦 Archive location: {RESULTS_ARCHIVE_BASE}")

archived_count = create_rerun_archive(
    RESULTS_ARCHIVE_BASE,
    RERUN_SESSION_ID,
    config,
    {
        'classification': CLASSIF_DIR,
        'ner': NER_DIR,
        'url_extraction': URL_DIR,
        'names': NAMES_DIR,
        'final': FINAL_DIR
    }
)

print(f"\n✅ Archive created with {archived_count} items")

# Final summary
completion_time = datetime.now()
processing_time = completion_time - datetime.fromisoformat(config['created'])
processing_mins = int(processing_time.total_seconds() / 60)
processing_hours = processing_mins // 60
processing_mins_remainder = processing_mins % 60

print("\n" + "🎉" * 20)
print("🎉 2022 INVENTORY RERUN COMPLETE")
print("🎉" * 20)

show_rerun_progress(progress, RERUN_SESSION_ID)

print(f"\n📊 Final Status:")
print(f"   Rerun Session ID: {RERUN_SESSION_ID}")
print(f"   Training Session Used: {training_session_used}")
print(f"   Completion Time: {completion_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Processing Time: {processing_hours}h {processing_mins_remainder}m")
print(f"   Papers Processed: {papers_to_process:,}")
print(f"   Final Inventory: {final_count:,} resources")

print(f"\n📁 Results Archive:")
print(f"   📁 {RESULTS_ARCHIVE_BASE}")
print(f"   📄 Main Output: final_inventory.csv")
print(f"   📄 Configuration: config_with_traceability.json")
print(f"   📄 Documentation: README.md")

print(f"\n🎯 Model Traceability:")
print(f"   Training Session → {training_session_used}")
print(f"   Rerun Session → {RERUN_SESSION_ID}")
print(f"   Models Source → {model_source}")
print(f"   Archive Location → {RESULTS_ARCHIVE_BASE}")

print(f"\n✨ Rerun completed successfully!")
print(f"🆔 Session ID: {RERUN_SESSION_ID} (save this for future reference)")
print(f"🔗 Training Session: {training_session_used} (linked models)")

print(f"\n🏁 Completed at {completion_time.strftime('%H:%M:%S')}")
print("📋 Full traceability chain established from training to final inventory!")